# HumanEval ANOVA and post-hoc comparison tables from saved per-run task results

This notebook loads the same saved per-run HumanEval task result CSV files used in the statistics notebook and creates **ANOVA-focused tables plus post-hoc group comparison tables**.

It reports:

1. group descriptives for run-level benchmark performance,
2. a standard one-way ANOVA table,
3. a Welch ANOVA table, which is preferable when group variances may differ and the number of runs is small,
4. variance assumption/context tables such as Levene/Brown-Forsythe checks,
5. Games-Howell post-hoc comparisons to identify where group differences occur,
6. pairwise effect sizes, including Cohen's d and Hedges' g.

No pass@k curves, plots, or task-level statistical tests are produced here.


In [ ]:
# =========================
# 1. Setup
# =========================
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd

try:
    from scipy import stats
except ImportError:
    raise ImportError('This notebook needs scipy. In Colab, run: !pip install scipy')

try:
    from statsmodels.stats.oneway import anova_oneway
except ImportError:
    raise ImportError('This notebook needs statsmodels. In Colab, run: !pip install statsmodels')


In [ ]:
# =========================
# 2. Mount Google Drive
# =========================
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# =========================
# 3. Paths
# =========================
PROJECT_DIR = Path('/content/drive/MyDrive/Speciale/4. Model Evaluation')
RESULTS_DIR = PROJECT_DIR / 'results'
TASK_RESULTS_DIR = RESULTS_DIR / 'per_run_task_results'

OUTPUT_DIR = RESULTS_DIR / 'anova_tables'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_RUN_COUNTS_CSV = OUTPUT_DIR / 'model_run_counts.csv'
RUN_LEVEL_SCORES_CSV = OUTPUT_DIR / 'run_level_scores_for_anova.csv'
RUN_LEVEL_DESCRIPTIVES_CSV = OUTPUT_DIR / 'run_level_anova_descriptives.csv'
STANDARD_ANOVA_CSV = OUTPUT_DIR / 'run_level_standard_oneway_anova.csv'
WELCH_ANOVA_CSV = OUTPUT_DIR / 'run_level_welch_anova.csv'
VARIANCE_CHECKS_CSV = OUTPUT_DIR / 'run_level_variance_assumption_checks.csv'
ANOVA_COMBINED_CSV = OUTPUT_DIR / 'run_level_anova_combined_summary.csv'
GAMES_HOWELL_CSV = OUTPUT_DIR / 'run_level_games_howell_posthoc.csv'
PAIRWISE_EFFECT_SIZES_CSV = OUTPUT_DIR / 'run_level_pairwise_effect_sizes.csv'
POSTHOC_COMBINED_CSV = OUTPUT_DIR / 'run_level_posthoc_combined_table.csv'

EXPECTED_RUNS_PER_MODEL = 10
ALPHA = 0.05

print('TASK_RESULTS_DIR:', TASK_RESULTS_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)


## Data loading assumptions

This notebook assumes each file in `TASK_RESULTS_DIR` corresponds to one model/run and follows the naming pattern:

```text
<model_label>__run_<run_idx>_task_results.csv
```

Each CSV must contain at least:

- `task_id`
- `passed`

The notebook computes one run-level score per model/run:

```text
pass1 = mean(passed)
```


In [ ]:
# =========================
# 4. Helper functions
# =========================
RUN_FILE_PATTERN = re.compile(r'^(?P<model_label>.+)__run_(?P<run_idx>\d+)_task_results\.csv$')


def parse_bool_like(x):
    if pd.isna(x):
        return False
    if isinstance(x, bool):
        return x
    if isinstance(x, (int, np.integer, float, np.floating)):
        return bool(x)
    s = str(x).strip().lower()
    true_values = {'true', '1', 'yes', 'y', 'passed', 'pass'}
    false_values = {'false', '0', 'no', 'n', 'failed', 'fail', ''}
    if s in true_values:
        return True
    if s in false_values:
        return False
    return bool(s)


def discover_task_result_files(task_results_dir: Path) -> pd.DataFrame:
    files = sorted(task_results_dir.glob('*_task_results.csv'))
    rows = []
    for path in files:
        m = RUN_FILE_PATTERN.match(path.name)
        rows.append({
            'file_path': str(path),
            'file_name': path.name,
            'inferred_model_label': m.group('model_label') if m else None,
            'inferred_run_idx': int(m.group('run_idx')) if m else None,
        })
    return pd.DataFrame(rows)


def read_one_run_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    required_cols = {'task_id', 'passed'}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f'{path.name} is missing required columns: {sorted(missing)}')

    df = df.copy()
    df['source_file'] = path.name
    df['source_path'] = str(path)

    m = RUN_FILE_PATTERN.match(path.name)
    inferred_model = m.group('model_label') if m else None
    inferred_run_idx = int(m.group('run_idx')) if m else None

    if 'model_label' not in df.columns:
        df['model_label'] = inferred_model
    else:
        df['model_label'] = df['model_label'].fillna(inferred_model)

    if 'run_idx' not in df.columns:
        df['run_idx'] = inferred_run_idx
    else:
        df['run_idx'] = df['run_idx'].fillna(inferred_run_idx)

    df['passed'] = df['passed'].apply(parse_bool_like)
    df['run_idx'] = pd.to_numeric(df['run_idx'], errors='coerce').astype('Int64')

    keep_cols = [c for c in [
        'task_id', 'model_label', 'run_idx', 'seed', 'passed',
        'result', 'source_file', 'source_path'
    ] if c in df.columns]

    df = df[keep_cols].copy()

    dupes = df.duplicated(subset=['task_id'], keep=False)
    if dupes.any():
        duped_tasks = df.loc[dupes, 'task_id'].tolist()[:10]
        raise ValueError(
            f'{path.name} contains duplicate task_id rows. Example duplicates: {duped_tasks}'
        )

    return df


def load_all_runs(task_results_dir: Path) -> pd.DataFrame:
    files = sorted(task_results_dir.glob('*_task_results.csv'))
    if not files:
        raise FileNotFoundError(f'No *_task_results.csv files found in {task_results_dir}')

    all_parts = []
    for path in files:
        all_parts.append(read_one_run_csv(path))

    all_runs = pd.concat(all_parts, ignore_index=True)

    if all_runs['model_label'].isna().any():
        bad_files = all_runs.loc[all_runs['model_label'].isna(), 'source_file'].unique().tolist()
        raise ValueError(f'Could not infer model_label for files: {bad_files}')

    if all_runs['run_idx'].isna().any():
        bad_files = all_runs.loc[all_runs['run_idx'].isna(), 'source_file'].unique().tolist()
        raise ValueError(f'Could not infer run_idx for files: {bad_files}')

    all_runs['run_idx'] = all_runs['run_idx'].astype(int)
    return all_runs


def validate_runs(all_runs: pd.DataFrame, expected_runs=10) -> pd.DataFrame:
    run_counts = (
        all_runs[['model_label', 'run_idx', 'source_file']]
        .drop_duplicates()
        .groupby('model_label')
        .agg(
            n_runs=('run_idx', 'nunique'),
            run_indices=('run_idx', lambda s: sorted(set(s))),
            n_files=('source_file', 'nunique'),
        )
        .reset_index()
        .sort_values('model_label')
    )

    incomplete = run_counts[run_counts['n_runs'] != expected_runs]
    if not incomplete.empty:
        warnings.warn(
            'Some models do not have the expected number of runs.
'
            + incomplete.to_string(index=False)
        )
    return run_counts


def compute_run_level_scores(all_runs: pd.DataFrame) -> pd.DataFrame:
    return (
        all_runs
        .groupby(['model_label', 'run_idx'], as_index=False)
        .agg(
            pass1=('passed', 'mean'),
            n_tasks=('task_id', 'nunique'),
            n_passed=('passed', 'sum'),
            source_file=('source_file', 'first'),
        )
        .sort_values(['model_label', 'run_idx'])
        .reset_index(drop=True)
    )


def run_level_descriptives(run_level_scores: pd.DataFrame) -> pd.DataFrame:
    desc = (
        run_level_scores
        .groupby('model_label', as_index=False)
        .agg(
            n_runs=('pass1', 'count'),
            mean_pass1=('pass1', 'mean'),
            sd_pass1=('pass1', 'std'),
            se_pass1=('pass1', lambda s: s.std(ddof=1) / np.sqrt(s.count()) if s.count() > 1 else np.nan),
            median_pass1=('pass1', 'median'),
            min_pass1=('pass1', 'min'),
            max_pass1=('pass1', 'max'),
            n_tasks_median=('n_tasks', 'median'),
        )
    )
    desc['ci95_low_mean_pass1'] = desc['mean_pass1'] - stats.t.ppf(0.975, desc['n_runs'] - 1) * desc['se_pass1']
    desc['ci95_high_mean_pass1'] = desc['mean_pass1'] + stats.t.ppf(0.975, desc['n_runs'] - 1) * desc['se_pass1']
    return desc.sort_values('mean_pass1', ascending=False).reset_index(drop=True)


def standard_oneway_anova_table(run_level_scores: pd.DataFrame) -> pd.DataFrame:
    groups = [
        g['pass1'].dropna().to_numpy(dtype=float)
        for _, g in run_level_scores.groupby('model_label')
    ]
    labels = sorted(run_level_scores['model_label'].unique())
    k = len(groups)
    n_total = sum(len(g) for g in groups)

    if k < 2:
        raise ValueError('Need at least two models for ANOVA.')
    if any(len(g) < 2 for g in groups):
        warnings.warn('At least one group has fewer than two observations. ANOVA may be invalid.')

    all_values = np.concatenate(groups)
    grand_mean = all_values.mean()

    ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
    ss_within = sum(((g - g.mean()) ** 2).sum() for g in groups)
    ss_total = ss_between + ss_within

    df_between = k - 1
    df_within = n_total - k
    df_total = n_total - 1

    ms_between = ss_between / df_between
    ms_within = ss_within / df_within
    f_stat = ms_between / ms_within
    p_value = stats.f.sf(f_stat, df_between, df_within)

    eta_squared = ss_between / ss_total if ss_total > 0 else np.nan
    omega_squared = (
        (ss_between - df_between * ms_within) / (ss_total + ms_within)
        if (ss_total + ms_within) > 0 else np.nan
    )

    return pd.DataFrame([
        {
            'source': 'between_models',
            'sum_sq': ss_between,
            'df': df_between,
            'mean_sq': ms_between,
            'F': f_stat,
            'p_value': p_value,
            'eta_squared': eta_squared,
            'omega_squared': omega_squared,
            'n_total': n_total,
            'n_groups': k,
        },
        {
            'source': 'within_models_error',
            'sum_sq': ss_within,
            'df': df_within,
            'mean_sq': ms_within,
            'F': np.nan,
            'p_value': np.nan,
            'eta_squared': np.nan,
            'omega_squared': np.nan,
            'n_total': n_total,
            'n_groups': k,
        },
        {
            'source': 'total',
            'sum_sq': ss_total,
            'df': df_total,
            'mean_sq': np.nan,
            'F': np.nan,
            'p_value': np.nan,
            'eta_squared': np.nan,
            'omega_squared': np.nan,
            'n_total': n_total,
            'n_groups': k,
        },
    ])


def welch_anova_table(run_level_scores: pd.DataFrame) -> pd.DataFrame:
    # statsmodels handles the Welch-Satterthwaite degrees of freedom.
    result = anova_oneway(
        run_level_scores['pass1'].to_numpy(dtype=float),
        groups=run_level_scores['model_label'].to_numpy(),
        use_var='unequal',
        welch_correction=True,
    )

    return pd.DataFrame([
        {
            'test': 'Welch ANOVA',
            'dependent_variable': 'run_level_pass1',
            'F': float(result.statistic),
            'df_num': float(result.df[0]),
            'df_denom': float(result.df[1]),
            'p_value': float(result.pvalue),
            'n_total': int(len(run_level_scores)),
            'n_groups': int(run_level_scores['model_label'].nunique()),
            'note': 'Welch ANOVA does not assume equal group variances.',
        }
    ])


def variance_assumption_checks(run_level_scores: pd.DataFrame) -> pd.DataFrame:
    groups = [
        g['pass1'].dropna().to_numpy(dtype=float)
        for _, g in run_level_scores.groupby('model_label')
    ]

    levene_mean = stats.levene(*groups, center='mean')
    brown_forsythe = stats.levene(*groups, center='median')

    return pd.DataFrame([
        {
            'test': 'Levene',
            'center': 'mean',
            'statistic': float(levene_mean.statistic),
            'p_value': float(levene_mean.pvalue),
            'interpretation': 'Small p-value suggests unequal variances.'
        },
        {
            'test': 'Brown-Forsythe',
            'center': 'median',
            'statistic': float(brown_forsythe.statistic),
            'p_value': float(brown_forsythe.pvalue),
            'interpretation': 'Median-centered variance check; often more robust than Levene with mean center.'
        },
    ])



def pairwise_effect_sizes_table(run_level_scores: pd.DataFrame) -> pd.DataFrame:
    """Pairwise mean differences and standardized effect sizes.

    Hedges' g is the preferred small-sample standardized effect size.
    mean_diff is group_a - group_b, so positive values mean group_a has higher pass1.
    """
    rows = []
    labels = sorted(run_level_scores['model_label'].dropna().unique())

    for i, group_a in enumerate(labels):
        a = run_level_scores.loc[run_level_scores['model_label'] == group_a, 'pass1'].dropna().to_numpy(dtype=float)
        for group_b in labels[i + 1:]:
            b = run_level_scores.loc[run_level_scores['model_label'] == group_b, 'pass1'].dropna().to_numpy(dtype=float)

            n_a, n_b = len(a), len(b)
            mean_a, mean_b = np.mean(a), np.mean(b)
            sd_a = np.std(a, ddof=1) if n_a > 1 else np.nan
            sd_b = np.std(b, ddof=1) if n_b > 1 else np.nan
            mean_diff = mean_a - mean_b

            pooled_sd = np.sqrt(((n_a - 1) * sd_a**2 + (n_b - 1) * sd_b**2) / (n_a + n_b - 2)) if n_a + n_b > 2 else np.nan
            cohen_d = mean_diff / pooled_sd if pooled_sd and pooled_sd > 0 else np.nan

            # Small-sample correction for Cohen's d.
            df = n_a + n_b - 2
            hedges_correction = 1 - (3 / (4 * df - 1)) if df > 1 else np.nan
            hedges_g = cohen_d * hedges_correction if pd.notna(cohen_d) and pd.notna(hedges_correction) else np.nan

            rows.append({
                'group_a': group_a,
                'group_b': group_b,
                'n_a': n_a,
                'n_b': n_b,
                'mean_a': mean_a,
                'mean_b': mean_b,
                'sd_a': sd_a,
                'sd_b': sd_b,
                'mean_diff_a_minus_b': mean_diff,
                'cohen_d': cohen_d,
                'hedges_g': hedges_g,
                'effect_size_note': 'Positive values mean group_a has higher run-level pass1 than group_b.'
            })

    return pd.DataFrame(rows).sort_values('mean_diff_a_minus_b', ascending=False).reset_index(drop=True)


def games_howell_posthoc_table(run_level_scores: pd.DataFrame, alpha: float = 0.05) -> pd.DataFrame:
    """Games-Howell post-hoc pairwise comparisons.

    This is a Welch-compatible post-hoc procedure: it does not assume equal variances
    and is therefore preferable to Tukey HSD when group variances may differ.
    """
    if not hasattr(stats, 'studentized_range'):
        raise ImportError(
            'scipy.stats.studentized_range is required for Games-Howell. '
            'Please upgrade scipy, e.g. !pip install --upgrade scipy'
        )

    labels = sorted(run_level_scores['model_label'].dropna().unique())
    k = len(labels)
    rows = []

    grouped = {}
    for label in labels:
        values = run_level_scores.loc[run_level_scores['model_label'] == label, 'pass1'].dropna().to_numpy(dtype=float)
        grouped[label] = {
            'values': values,
            'n': len(values),
            'mean': np.mean(values),
            'var': np.var(values, ddof=1) if len(values) > 1 else np.nan,
            'sd': np.std(values, ddof=1) if len(values) > 1 else np.nan,
        }

    for i, group_a in enumerate(labels):
        a = grouped[group_a]
        for group_b in labels[i + 1:]:
            b = grouped[group_b]

            n_a, n_b = a['n'], b['n']
            mean_diff = a['mean'] - b['mean']
            se = np.sqrt(a['var'] / n_a + b['var'] / n_b)

            numerator = (a['var'] / n_a + b['var'] / n_b) ** 2
            denominator = ((a['var'] / n_a) ** 2 / (n_a - 1)) + ((b['var'] / n_b) ** 2 / (n_b - 1))
            df = numerator / denominator if denominator > 0 else np.nan

            t_stat = abs(mean_diff) / se if se > 0 else np.nan
            q_stat = t_stat * np.sqrt(2) if pd.notna(t_stat) else np.nan
            p_value = stats.studentized_range.sf(q_stat, k, df) if pd.notna(q_stat) and pd.notna(df) else np.nan

            q_crit = stats.studentized_range.ppf(1 - alpha, k, df) if pd.notna(df) else np.nan
            ci_half_width = q_crit * se / np.sqrt(2) if pd.notna(q_crit) and pd.notna(se) else np.nan

            rows.append({
                'group_a': group_a,
                'group_b': group_b,
                'n_a': n_a,
                'n_b': n_b,
                'mean_a': a['mean'],
                'mean_b': b['mean'],
                'mean_diff_a_minus_b': mean_diff,
                'se': se,
                't_stat_abs': t_stat,
                'df_welch_satterthwaite': df,
                'p_value_games_howell': p_value,
                'ci95_low_mean_diff': mean_diff - ci_half_width,
                'ci95_high_mean_diff': mean_diff + ci_half_width,
                'significant_alpha_0_05': bool(p_value < alpha) if pd.notna(p_value) else False,
                'direction': f'{group_a} > {group_b}' if mean_diff > 0 else f'{group_a} < {group_b}' if mean_diff < 0 else 'equal means',
                'test_note': 'Games-Howell post-hoc comparison; positive mean_diff means group_a has higher pass1.'
            })

    return pd.DataFrame(rows).sort_values('p_value_games_howell').reset_index(drop=True)


def combined_posthoc_table(games_howell: pd.DataFrame, effect_sizes: pd.DataFrame) -> pd.DataFrame:
    merged = games_howell.merge(
        effect_sizes[['group_a', 'group_b', 'cohen_d', 'hedges_g']],
        on=['group_a', 'group_b'],
        how='left'
    )

    preferred_cols = [
        'group_a', 'group_b',
        'mean_a', 'mean_b', 'mean_diff_a_minus_b',
        'p_value_games_howell', 'significant_alpha_0_05',
        'ci95_low_mean_diff', 'ci95_high_mean_diff',
        'hedges_g', 'cohen_d',
        'df_welch_satterthwaite', 'se', 't_stat_abs',
        'n_a', 'n_b', 'direction', 'test_note'
    ]
    return merged[preferred_cols].sort_values('p_value_games_howell').reset_index(drop=True)


In [ ]:
# =========================
# 5. Discover and load files
# =========================
files_df = discover_task_result_files(TASK_RESULTS_DIR)
print(f'Found {len(files_df)} task-result files.')
display(files_df.head(20))

all_runs = load_all_runs(TASK_RESULTS_DIR)
print('Loaded rows:', len(all_runs))
print('Models:', sorted(all_runs['model_label'].unique().tolist()))
display(all_runs.head())


In [ ]:
# =========================
# 6. Validate run coverage
# =========================
run_counts = validate_runs(all_runs, expected_runs=EXPECTED_RUNS_PER_MODEL)
display(run_counts)

run_counts.to_csv(MODEL_RUN_COUNTS_CSV, index=False)
print('Saved:', MODEL_RUN_COUNTS_CSV)


In [ ]:
# =========================
# 7. Compute run-level pass@1 scores
# =========================
run_level_scores = compute_run_level_scores(all_runs)

display(run_level_scores.head(20))

run_level_scores.to_csv(RUN_LEVEL_SCORES_CSV, index=False)
print('Saved:', RUN_LEVEL_SCORES_CSV)


## ANOVA tables

The standard one-way ANOVA table is included for familiarity, but the Welch ANOVA table should generally be treated as the safer main ANOVA result when there are only around 10 runs per model and equal variances are uncertain.


In [ ]:
# =========================
# 8. Group descriptives
# =========================
descriptives = run_level_descriptives(run_level_scores)
display(descriptives)

descriptives.to_csv(RUN_LEVEL_DESCRIPTIVES_CSV, index=False)
print('Saved:', RUN_LEVEL_DESCRIPTIVES_CSV)


In [ ]:
# =========================
# 9. Standard one-way ANOVA table
# =========================
standard_anova = standard_oneway_anova_table(run_level_scores)
display(standard_anova)

standard_anova.to_csv(STANDARD_ANOVA_CSV, index=False)
print('Saved:', STANDARD_ANOVA_CSV)


In [ ]:
# =========================
# 10. Welch ANOVA table
# =========================
welch_anova = welch_anova_table(run_level_scores)
display(welch_anova)

welch_anova.to_csv(WELCH_ANOVA_CSV, index=False)
print('Saved:', WELCH_ANOVA_CSV)


In [ ]:
# =========================
# 11. Variance assumption/context checks
# =========================
variance_checks = variance_assumption_checks(run_level_scores)
display(variance_checks)

variance_checks.to_csv(VARIANCE_CHECKS_CSV, index=False)
print('Saved:', VARIANCE_CHECKS_CSV)


In [ ]:
# =========================
# 12. Combined ANOVA summary table
# =========================
standard_main = standard_anova.loc[standard_anova['source'] == 'between_models'].copy()
standard_summary = pd.DataFrame([
    {
        'test': 'Standard one-way ANOVA',
        'dependent_variable': 'run_level_pass1',
        'F': float(standard_main['F'].iloc[0]),
        'df_num': float(standard_main['df'].iloc[0]),
        'df_denom': float(standard_anova.loc[standard_anova['source'] == 'within_models_error', 'df'].iloc[0]),
        'p_value': float(standard_main['p_value'].iloc[0]),
        'eta_squared': float(standard_main['eta_squared'].iloc[0]),
        'omega_squared': float(standard_main['omega_squared'].iloc[0]),
        'n_total': int(standard_main['n_total'].iloc[0]),
        'n_groups': int(standard_main['n_groups'].iloc[0]),
        'note': 'Assumes equal group variances.'
    }
])

welch_summary = welch_anova.copy()
welch_summary['eta_squared'] = np.nan
welch_summary['omega_squared'] = np.nan

combined_anova_summary = pd.concat(
    [standard_summary, welch_summary[standard_summary.columns]],
    ignore_index=True
)

display(combined_anova_summary)

combined_anova_summary.to_csv(ANOVA_COMBINED_CSV, index=False)
print('Saved:', ANOVA_COMBINED_CSV)


## Post-hoc tables: where are the group differences?

The ANOVA tables above answer whether there is evidence of any difference across models/methodologies. They do not identify which groups differ.

For that, this notebook uses **Games-Howell post-hoc comparisons**, which are a good companion to Welch ANOVA because they do not assume equal variances. The combined post-hoc table also includes Cohen's d and **Hedges' g**, with Hedges' g preferred for small sample sizes.


In [ ]:
# =========================
# 13. Games-Howell post-hoc pairwise comparisons
# =========================
games_howell = games_howell_posthoc_table(run_level_scores, alpha=ALPHA)
display(games_howell)

games_howell.to_csv(GAMES_HOWELL_CSV, index=False)
print('Saved:', GAMES_HOWELL_CSV)


In [ ]:
# =========================
# 14. Pairwise effect sizes
# =========================
pairwise_effect_sizes = pairwise_effect_sizes_table(run_level_scores)
display(pairwise_effect_sizes)

pairwise_effect_sizes.to_csv(PAIRWISE_EFFECT_SIZES_CSV, index=False)
print('Saved:', PAIRWISE_EFFECT_SIZES_CSV)


In [ ]:
# =========================
# 15. Combined post-hoc comparison table
# =========================
posthoc_combined = combined_posthoc_table(games_howell, pairwise_effect_sizes)
display(posthoc_combined)

posthoc_combined.to_csv(POSTHOC_COMBINED_CSV, index=False)
print('Saved:', POSTHOC_COMBINED_CSV)


## Suggested interpretation

For reporting, prefer the Welch ANOVA result when the variance checks suggest unequal variance or when you want a conservative default for small samples.

Use the Games-Howell table to describe **where** the group differences occur after the omnibus ANOVA. Report Hedges' g alongside p-values so the result is not interpreted only through statistical significance.

A concise thesis-style sentence could be:

> Differences in run-level HumanEval pass@1 across methodologies were assessed using Welch's one-way ANOVA because it does not assume equal group variances. Where the omnibus test indicated differences, Games-Howell post-hoc comparisons were used to identify pairwise differences, with Hedges' g reported as a small-sample effect size.
